In [10]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import sys
import shutil
import importlib
from dataclasses import fields
import json
from datetime import datetime

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


# Notebook location:
# THESIS/z.parcels_postprocessing/notebooks/run_particles_mr_sm.ipynb
NB_DIR = Path.cwd().resolve()
PROJECT_DIR = NB_DIR.parent.parent

# Add the project root and the two postprocessing folders.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if str(PROJECT_DIR / "z.flow_postprocessing") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "z.flow_postprocessing"))
if str(PROJECT_DIR / "z.parcels_postprocessing") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "z.parcels_postprocessing"))


# Project modules
import scripts.fieldset as pfs
import scripts.particles as pparticles
import scripts.kernels_common as kcommon
import scripts.kernels_passive as kpassive
import scripts.kernels_mr_sm as kmrsm
import scripts.run as prun
import scripts.notebook_helpers as phelp
import theme.plot_theme as ptheme


for module in [
    pfs,
    pparticles,
    kcommon,
    kpassive,
    kmrsm,
    prun,
    phelp,
    ptheme,
]:
    importlib.reload(module)

ptheme.apply_theme()

print(f"Notebook dir : {NB_DIR}")
print(f"Project dir  : {PROJECT_DIR}")
print(f"Parcels dir  : {PROJECT_DIR / 'z.parcels_postprocessing'}")
print(f"run.py loaded from: {Path(prun.__file__).resolve()}")
print(f"passive kernel loaded from: {Path(kpassive.__file__).resolve()}")
print(f"MR-SM kernel loaded from: {Path(kmrsm.__file__).resolve()}")


Notebook dir : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\notebooks
Project dir  : C:\Users\Jelle Gortemaker\Documents\Thesis
Parcels dir  : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing
run.py loaded from: C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\scripts\run.py
passive kernel loaded from: C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\scripts\kernels_passive.py
MR-SM kernel loaded from: C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\scripts\kernels_mr_sm.py


C:\Users\Jelle Gortemaker\Documents\Thesis\theme\plot_theme.py:110: UserWarning: Overwriting the cmap 'toc' that was already in the registry.
  mpl.colormaps.register(cmap, name=name, force=True)
C:\Users\Jelle Gortemaker\Documents\Thesis\theme\plot_theme.py:110: UserWarning: Overwriting the cmap 'toc_cmap' that was already in the registry.
  mpl.colormaps.register(cmap, name=name, force=True)
C:\Users\Jelle Gortemaker\Documents\Thesis\theme\plot_theme.py:110: UserWarning: Overwriting the cmap 'toc_cmap_r' that was already in the registry.
  mpl.colormaps.register(cmap, name=name, force=True)


In [11]:
# ============================================================
# 2. DIRECTORY / CASE SETTINGS
# ============================================================

case_name = "run_jun1"   # edit only this
RUN_TITLE_INFO = "Case JUN-1 Production"  # keep compact; used only for metadata

INPUT_DIR = (NB_DIR / "../data/input").resolve()
DATA_FILE = (INPUT_DIR / f"{case_name}.nc").resolve()

if not DATA_FILE.exists():
    available = sorted(p.name for p in INPUT_DIR.glob("*"))
    raise FileNotFoundError(
        f"Could not find input file:\n{DATA_FILE}\n\n"
        f"Available files in {INPUT_DIR}:\n"
        + "\n".join(available)
    )


# ============================================================
# 3. PARCELS ADVECTION SETTINGS
# ============================================================

DAY_PER_INDEX = 3.0 / 24.0
TIME_STEP_SECONDS = int(DAY_PER_INDEX * 86400)

RELEASE_TIME_INDEX = 0
LEVEL_INDICES = (0,)

RUNTIME_MODE = "full_forcing"       # "manual" or "full_forcing"
# RUNTIME_DAYS_REQUESTED = 9.0

DT_SECONDS = 300
OUTPUTDT_SECONDS = TIME_STEP_SECONDS

NX = 60
NY = 60

PERIODIC = True
RELEASE_MARGIN_CELLS = 1.0

RUN_ADVECTION = True
OVERWRITE_OUTPUT = True

# For a separate analysis notebook, keep this True.
SAVE_TRAJECTORIES = True
SAVE_METADATA = True


# ============================================================
# 4. PARTICLE CLASS SETTINGS
# ============================================================

# Explicit Coriolis is disabled
F0 = 0.0

# extract eddy turnover times
ETT_CASE_NAME = case_name

ETT_JSON = (
    PROJECT_DIR
    / "OGCM"
    / "data"
    / "processed"
    / f"ETT_{ETT_CASE_NAME}.json"
)

if not ETT_JSON.exists():
    available_ett_files = sorted(
        p.name
        for p in (PROJECT_DIR / "OGCM" / "data" / "processed").glob("ETT_*.json")
    )

    raise FileNotFoundError(
        f"Could not find eddy-turnover-time metadata:\n{ETT_JSON}\n\n"
        "Available ETT files:\n"
        + "\n".join(available_ett_files)
    )

with open(ETT_JSON, "r") as f:
    ett_metadata = json.load(f)

try:
    MEDIAN_EDDY_TURNOVER_DAYS = float(
        ett_metadata["eddy_turnover_time_days"]["median_T_eddy_days"]
    )
except KeyError as exc:
    raise KeyError(
        "Expected the following entry in the ETT JSON:\n"
        "eddy_turnover_time_days -> median_T_eddy_days"
    ) from exc


# User-selected characteristic flow timescale:
#
#     T_flow = 0.5 * median eddy turnover time
#
FLOW_TIMESCALE_DAYS = 0.5 * MEDIAN_EDDY_TURNOVER_DAYS
FLOW_TIMESCALE_SECONDS = FLOW_TIMESCALE_DAYS * 86400

print("\nCharacteristic flow timescale")
print(f"  source JSON           : {ETT_JSON}")
print(f"  median T_eddy         : {MEDIAN_EDDY_TURNOVER_DAYS:.3f} days")
print(f"  selected T_flow       : {FLOW_TIMESCALE_DAYS:.3f} days")
print(f"  selected T_flow       : {FLOW_TIMESCALE_SECONDS:.3e} s")


# ------------------------------------------------------------
# Prescribed effective relaxation times
# ------------------------------------------------------------

TAU_P_SECONDS_LIST = [
    12.0 * 3600.0,
    # Add further cases here.
]


particle_specs = phelp.build_particle_specs(
    tau_p_seconds_list=TAU_P_SECONDS_LIST,
    include_passive=True,
    flow_timescale_seconds=FLOW_TIMESCALE_SECONDS,
)

phelp.print_particle_specs(particle_specs)


# ============================================================
# 5. OUTPUT ORGANIZATION
# results/<case_name>/...
# ============================================================

release_time_days = RELEASE_TIME_INDEX * DAY_PER_INDEX
level_tag = "k" + "-".join(str(k) for k in LEVEL_INDICES)

has_mr_sm = any(spec["particle_class"] == "mr_sm" for spec in particle_specs)
particle_collection_tag = "passive_plus_mrsm" if has_mr_sm else "passive"
run_collection_id = f"{particle_collection_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}"

paths = phelp.prepare_output_paths(
    notebook_dir=NB_DIR,
    case_name=case_name,
    run_collection_id=run_collection_id,
    save_trajectories=SAVE_TRAJECTORIES,
    save_metadata=SAVE_METADATA,
    save_figures=False,
    compute_statistics=False,
)

CASE_RESULTS_DIR = paths.case_results_dir
RUN_OUT_DIR = paths.run_out_dir
METADATA_DIR = paths.metadata_dir
CONFIG_JSON = paths.config_json

phelp.print_output_paths(
    DATA_FILE,
    INPUT_DIR,
    paths,
    save_trajectories=SAVE_TRAJECTORIES,
    save_figures=False,
)

print(f"\nRun collection ID: {run_collection_id}")
print(f"Collection config: {CONFIG_JSON}")



Characteristic flow timescale
  source JSON           : C:\Users\Jelle Gortemaker\Documents\Thesis\OGCM\data\processed\ETT_run_jun1.json
  median T_eddy         : 16.479 days
  selected T_flow       : 8.239 days
  selected T_flow       : 7.119e+05 s

Particle classes
  passive | Passive particles | tau_p=0 s
  tau12h | τₚ=12 h, St=0.0607 | tau_p=43200 s

Input / output
  input file        : run_jun1.nc
  input dir         : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\data\input
  case results dir  : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1
  trajectory dir    : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories
  save trajectories : True
  figure dir        : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\figures\trajectory_snapshots
  save figures      : False

Run collection ID: passive_plus_mrsm_k0_release_t0000
Collection config: 

In [12]:
# ============================================================
# 6. CHECK RUNCONFIG AND INPUT FIELD
# ============================================================

required_runconfig_fields = {
    "input_nc",
    "output_path",
    "runtime_days",
    "dt_seconds",
    "outputdt_seconds",
    "time_step_seconds",
    "release_time_index",
    "periodic",
    "level_indices",
    "release_margin_cells",
    "particle_class",
    "particle_tag",
    "particle_label",
    "tau_p_seconds",
    "f0",
    "flow_timescale_seconds",
}

available_runconfig_fields = {f.name for f in fields(prun.RunConfig)}
missing = required_runconfig_fields - available_runconfig_fields

if missing:
    raise RuntimeError(
        "RunConfig is missing required tau-only fields:\n"
        f"{sorted(missing)}\n\n"
        "Update scripts/run.py with tau_p_seconds, then restart the kernel or reload prun."
    )

# Optional warning only: these old fields should no longer be used in the notebook.
deprecated_fields = {
    "B",
    "diameter_m",
    "nu_m2_s",
    "drag_correction",
    "C_Rep",
    "Rep_max",
}

still_present = deprecated_fields & available_runconfig_fields
if still_present:
    print(
        "Note: old diameter/buoyancy/drag fields still exist in RunConfig, "
        "but this notebook will not use them:"
    )
    print(f"  {sorted(still_present)}")


raw_qc = pfs.summarize_dataset(DATA_FILE)

print("\nRaw MITgcm input QC")
print(f"  dims      : {raw_qc.get('dims')}")
print(f"  U dims    : {raw_qc.get('u_dims', 'not reported')}")
print(f"  V dims    : {raw_qc.get('v_dims', 'not reported')}")
print(f"  x center  : {raw_qc.get('x_center', raw_qc.get('x_name', 'unknown'))}")
print(f"  y center  : {raw_qc.get('y_center', raw_qc.get('y_name', 'unknown'))}")
print(f"  U var     : {raw_qc.get('u_name')}")
print(f"  V var     : {raw_qc.get('v_name')}")


# Build a lightweight fieldset for QC/runtime selection.
# No derivative fields are needed for this QC check.
# Derivatives are built inside prun.run_parcels_experiment only for MR-SM runs.
fieldset_qc, meta, ds_parcels = pfs.build_fieldset(
    DATA_FILE,
    surface_only=True,
    mesh="flat",
    level_indices=LEVEL_INDICES,
    time_step_seconds=TIME_STEP_SECONDS,
    periodic=PERIODIC,
    add_derivatives=False,
)

qc = pfs.quick_qc_parcels_input(ds_parcels)

print("\nParcels-ready field QC")
print(f"  dims      : {qc.get('dims')}")
print(f"  U shape   : {qc.get('U_shape', qc.get('u_shape'))}")
print(f"  V shape   : {qc.get('V_shape', qc.get('v_shape'))}")
print(f"  x range   : {qc.get('x_min'):.1f} to {qc.get('x_max'):.1f} m")
print(f"  y range   : {qc.get('y_min'):.1f} to {qc.get('y_max'):.1f} m")
print(f"  n time    : {qc.get('n_time', ds_parcels.sizes['time'])}")
print(f"  dt head   : {np.diff(ds_parcels['time'].values[:5])}")
print(f"  U NaN t0  : {qc.get('U_nan_fraction_t0', qc.get('u_nan_fraction_t0'))}")
print(f"  V NaN t0  : {qc.get('V_nan_fraction_t0', qc.get('v_nan_fraction_t0'))}")


# ============================================================
# 7. RUNTIME SELECTION
# ============================================================

n_forcing_times = ds_parcels.sizes["time"]

max_runtime_days = (
    (n_forcing_times - 1 - RELEASE_TIME_INDEX)
    * TIME_STEP_SECONDS
    / 86400.0
)

if RELEASE_TIME_INDEX < 0 or RELEASE_TIME_INDEX >= n_forcing_times:
    raise ValueError(
        f"RELEASE_TIME_INDEX={RELEASE_TIME_INDEX} is outside the available forcing range "
        f"0 to {n_forcing_times - 1}."
    )

if RUNTIME_MODE == "full_forcing":
    RUNTIME_DAYS = max_runtime_days

elif RUNTIME_MODE == "manual":
    RUNTIME_DAYS = float(RUNTIME_DAYS_REQUESTED)

    if RUNTIME_DAYS > max_runtime_days:
        raise ValueError(
            f"Requested runtime is too long.\n"
            f"Requested runtime : {RUNTIME_DAYS:.3f} days\n"
            f"Available runtime : {max_runtime_days:.3f} days\n"
            f"Use RUNTIME_MODE='full_forcing' or reduce RUNTIME_DAYS_REQUESTED."
        )

else:
    raise ValueError(f"Unknown RUNTIME_MODE: {RUNTIME_MODE}")


print("\nRuntime settings")
print(f"  forcing time steps    : {n_forcing_times}")
print(f"  forcing dt            : {TIME_STEP_SECONDS} s")
print(f"  release index         : {RELEASE_TIME_INDEX}")
print(f"  max available runtime : {max_runtime_days:.3f} days")
print(f"  selected runtime      : {RUNTIME_DAYS:.3f} days")


print("\nParticle model convention")
print("  Passive tracer : tau_p_seconds = 0")
print("  MR-SM particle : tau_p_seconds is prescribed directly")
print("  Diameter, buoyancy, viscosity, and drag correction are not used by default.")


Raw MITgcm input QC
  dims      : {'Zmd000002': 2, 'T': 72, 'Y': 512, 'Xp1': 513, 'Yp1': 513, 'X': 512}
  U dims    : ('T', 'Zmd000002', 'Y', 'Xp1')
  V dims    : ('T', 'Zmd000002', 'Yp1', 'X')
  x center  : X
  y center  : Y
  U var     : UVEL
  V var     : VVEL

Parcels-ready field QC
  dims      : {'time': 72, 'y': 512, 'x': 512}
  U shape   : (72, 512, 512)
  V shape   : (72, 512, 512)
  x range   : 250.0 to 255750.0 m
  y range   : 250.0 to 255750.0 m
  n time    : 72
  dt head   : [10800. 10800. 10800. 10800.]
  U NaN t0  : 0.0
  V NaN t0  : 0.0

Runtime settings
  forcing time steps    : 72
  forcing dt            : 10800 s
  release index         : 0
  max available runtime : 8.875 days
  selected runtime      : 8.875 days

Particle model convention
  Passive tracer : tau_p_seconds = 0
  MR-SM particle : tau_p_seconds is prescribed directly
  Diameter, buoyancy, viscosity, and drag correction are not used by default.


In [13]:
# ============================================================
# 8. PARCELS ADVECTION
# One trajectory file per particle class.
# ============================================================

run_outputs = {}

for spec in particle_specs:
    particle_tag = spec["tag"]
    particle_label = spec.get("label", particle_tag)
    particle_class = spec["particle_class"]

    tau_p_seconds = float(spec.get("tau_p_seconds", 0.0))

    if particle_class == "mr_sm" and tau_p_seconds <= 0.0:
        raise ValueError(
            f"MR-SM particle '{particle_tag}' needs tau_p_seconds > 0.\n"
            f"Current spec:\n{spec}"
        )

    if particle_class == "passive":
        tau_p_seconds = 0.0

    run_id = f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}"
    out_zarr = RUN_OUT_DIR / f"{run_id}.zarr"
    metadata_json = out_zarr.with_suffix(".json")

    if OVERWRITE_OUTPUT and RUN_ADVECTION and out_zarr.exists():
        if out_zarr.is_dir():
            shutil.rmtree(out_zarr)
        else:
            out_zarr.unlink()

    if OVERWRITE_OUTPUT and RUN_ADVECTION and metadata_json.exists():
        metadata_json.unlink()

    if RUN_ADVECTION:
        config = prun.RunConfig(
            input_nc=str(DATA_FILE),
            output_path=str(out_zarr),

            runtime_days=RUNTIME_DAYS,
            dt_seconds=DT_SECONDS,
            outputdt_seconds=OUTPUTDT_SECONDS,
            time_step_seconds=TIME_STEP_SECONDS,
            release_time_index=RELEASE_TIME_INDEX,

            surface_only=True,
            mesh="flat",
            periodic=PERIODIC,
            level_indices=LEVEL_INDICES,

            release_mode="grid",
            nx=NX,
            ny=NY,
            release_margin_cells=RELEASE_MARGIN_CELLS,

            particle_class=particle_class,
            particle_tag=particle_tag,
            particle_label=particle_label,

            # Tau-only MR-SM control.
            # Passive particles use tau_p_seconds = 0.
            # MR-SM particles use the manually prescribed value.
            tau_p_seconds=tau_p_seconds,

            # Needed by the MR-SM kernel.
            f0=F0,

            # Only used for storing/reporting St = tau_p / T_flow.
            flow_timescale_seconds=spec.get(
                "flow_timescale_seconds",
                FLOW_TIMESCALE_SECONDS,
            ),

            save_metadata_sidecar=True,
        )

        print(f"\nRunning {particle_tag}")
        print(f"  class : {particle_class}")
        print(f"  label : {particle_label}")
        print(f"  tau_p : {tau_p_seconds:.3g} s")

        info = prun.run_parcels_experiment(config)

    else:
        if not out_zarr.exists():
            raise FileNotFoundError(
                f"RUN_ADVECTION=False, but output does not exist:\n{out_zarr}"
            )

        if metadata_json.exists():
            info = prun.load_run_metadata(metadata_json)
        else:
            info = {
                "particle_tag": particle_tag,
                "particle_label": particle_label,
                "particle_class": particle_class,
                "tau_p_seconds": tau_p_seconds,
                "output_path": str(out_zarr),
            }

    run_outputs[particle_tag] = {
        "path": str(out_zarr),
        "metadata_path": str(metadata_json),
        "label": info.get("particle_label", particle_label),
        "info": info,
        "spec": spec,
    }

    print(f"  output   : {out_zarr}")
    print(f"  metadata : {metadata_json}")


# ============================================================
# 9. SAVE COLLECTION METADATA
# ============================================================

def _json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {k: _json_ready(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [_json_ready(v) for v in obj]

    if isinstance(obj, np.generic):
        return obj.item()

    if isinstance(obj, float) and not np.isfinite(obj):
        return None

    return obj


flow_timescale_seconds_json = (
    None
    if FLOW_TIMESCALE_SECONDS is None
    else float(FLOW_TIMESCALE_SECONDS)
)

config_dict = {
    # --------------------------------------------------------
    # Collection identification
    # --------------------------------------------------------
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "case_name": case_name,
    "run_collection_id": run_collection_id,

    # --------------------------------------------------------
    # Input/output locations
    # --------------------------------------------------------
    "input_nc": str(DATA_FILE),
    "case_results_dir": str(CASE_RESULTS_DIR),
    "trajectory_dir": str(RUN_OUT_DIR),

    # --------------------------------------------------------
    # Release configuration
    # --------------------------------------------------------
    "release_time_index": int(RELEASE_TIME_INDEX),
    "release_time_days": float(release_time_days),
    "level_indices": list(LEVEL_INDICES),
    "nx": int(NX),
    "ny": int(NY),
    "periodic": bool(PERIODIC),
    "release_margin_cells": float(RELEASE_MARGIN_CELLS),
    "runtime_mode": RUNTIME_MODE,
    "runtime_days": float(RUNTIME_DAYS),
    "dt_seconds": int(DT_SECONDS),
    "outputdt_seconds": int(OUTPUTDT_SECONDS),
    "time_step_seconds": int(TIME_STEP_SECONDS),


    "particle_convention": (
        "Passive particles use tau_p_seconds=0. "
        "Inertial particles use a manually prescribed effective response time "
        "tau_p_seconds. The inertial correction is tau_p times the material "
        "acceleration of the fluid. Explicit Coriolis, diameter, buoyancy, "
        "viscosity, particle Reynolds number, and drag correction are not included."
    ),

    "tau_p_interpretation": (
        "Effective response coefficient multiplying the fluid material acceleration."
    ),

    "explicit_coriolis": False,
    "f0": float(F0),
    "eddy_turnover_case_name": ETT_CASE_NAME,
    "eddy_turnover_time_json": str(ETT_JSON),

    "median_eddy_turnover_time_days": float(
        MEDIAN_EDDY_TURNOVER_DAYS
    ),

    "flow_timescale_multiplier": 0.5,

    "flow_timescale_definition": (
        "T_flow = 0.5 * median_T_eddy_days"
    ),

    "flow_timescale_days": float(FLOW_TIMESCALE_DAYS),
    "flow_timescale_seconds": float(FLOW_TIMESCALE_SECONDS),

    "stokes_number_definition": (
        "St = tau_p_seconds / flow_timescale_seconds"
    ),
    "particle_tag_convention": (
        "passive for tracers; tau{value}{unit} for inertial particles"
    ),
    "particle_specs": particle_specs,
    "run_outputs": run_outputs,
}

if SAVE_METADATA:
    with open(CONFIG_JSON, "w") as f:
        json.dump(_json_ready(config_dict), f, indent=2)

    print(f"\nSaved collection metadata: {CONFIG_JSON}")

print("\nDone. Use this in the analysis notebook:")
print(f"RUN_COLLECTION_ID = {run_collection_id!r}")


Running passive
  class : passive
  label : Passive particles
  tau_p : 0 s


c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\ast.py:407: KernelWarning: Don't change the location of a particle directly in a Kernel. Use particle_dlon, particle_dlat, etc.
  return visitor(node)
c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\field.py:1184: RuntimeWarning: Sampling of velocities should normally be done using fieldset.UV or fieldset.UVW object; tread carefully
  self._check_velocitysampling()


INFO: Output files are stored in C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories\passive_k0_release_t0000.zarr.
  0%|          | 0/766800.0 [00:00<?, ?it/s]

c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\particledata.py:358: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  np.less_equal(time - np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\particledata.py:359: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  & np.greater_equal(time + np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\particledata.py:360: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  | ((np.isnan(pd["dt"])) & np.equal(time, pd["time"], where=np.isfinite(pd["time"])))


100%|██████████| 766800.0/766800.0 [00:52<00:00, 14678.10it/s]
  output   : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories\passive_k0_release_t0000.zarr
  metadata : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories\passive_k0_release_t0000.json

Running tau12h
  class : mr_sm
  label : τₚ=12 h, St=0.0607
  tau_p : 4.32e+04 s


c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\ast.py:407: KernelWarning: Don't change the location of a particle directly in a Kernel. Use particle_dlon, particle_dlat, etc.
  return visitor(node)
c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\field.py:1184: RuntimeWarning: Sampling of velocities should normally be done using fieldset.UV or fieldset.UVW object; tread carefully
  self._check_velocitysampling()


INFO: Output files are stored in C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories\tau12h_k0_release_t0000.zarr.
  0%|          | 0/766800.0 [00:00<?, ?it/s]

c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\particledata.py:358: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  np.less_equal(time - np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\particledata.py:359: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  & np.greater_equal(time + np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
c:\Users\Jelle Gortemaker\miniconda3\envs\thesis_parcels\Lib\site-packages\parcels\particledata.py:360: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  | ((np.isnan(pd["dt"])) & np.equal(time, pd["time"], where=np.isfinite(pd["time"])))


100%|██████████| 766800.0/766800.0 [01:41<00:00, 7566.71it/s]
  output   : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories\tau12h_k0_release_t0000.zarr
  metadata : C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\trajectories\tau12h_k0_release_t0000.json

Saved collection metadata: C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\results\run_jun1\metadata\passive_plus_mrsm_k0_release_t0000_config.json

Done. Use this in the analysis notebook:
RUN_COLLECTION_ID = 'passive_plus_mrsm_k0_release_t0000'


C:\Users\Jelle Gortemaker\AppData\Local\Temp\ipykernel_21900\841070500.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc": datetime.utcnow().isoformat() + "Z",
